In [86]:
import pandas as pd
import requests
import json
import numpy as np
from Bio import Entrez
from sklearn.preprocessing import MinMaxScaler
from rapidfuzz import process

In [90]:
# Load dataset

# sample dataset from pubmed
papers = pd.read_csv("sample_dataset.csv")
# sjr dataset to get sjr IF
sjr_data = pd.read_csv("scimagojr 2023.csv", sep=";")  # Adjust separator if needed
# Convert journal names to lowercase and remove whitespace
papers["Journal/Book"] = papers["Journal/Book"].str.lower().str.strip()
sjr_data["Title"] = sjr_data["Title"].str.lower().str.strip()
papers

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI
0,39142261,Data-driven mathematical modeling approaches f...,"Demongeot J, Magal P.",Phys Life Rev. 2024 Sep;50:166-208. doi: 10.10...,Demongeot J,phys life rev,2024,2024/08/14,NaN,NaN,10.1016/j.plrev.2024.08.004
1,38421182,Pathogenesis of viral infections during pregnancy,"Creisher PS, Klein SL.",Clin Microbiol Rev. 2024 Jun 13;37(2):e0007323...,Creisher PS,clin microbiol rev,2024,2024/02/29,PMC11237665,NaN,10.1128/cmr.00073-23
2,38230499,Recent advances in breast cancer cell line res...,"Sharma MP, Shukla S, Misra G.",Int J Cancer. 2024 May 15;154(10):1683-1693. d...,Sharma MP,int j cancer,2024,2024/01/17,NaN,NaN,10.1002/ijc.34849
3,39270616,Early mathematical models of COVID-19 vaccinat...,"Burch E, Khan SA, Stone J, Asgharzadeh A, Dawe...",Public Health. 2024 Nov;236:207-215. doi: 10.1...,Burch E,public health,2024,2024/09/13,NaN,NaN,10.1016/j.puhe.2024.07.029
4,38988830,Learning from the COVID-19 pandemic: A systema...,"González-Parra G, Mahmud MS, Kadelka C.",Infect Dis Model. 2024 May 15;9(4):1057-1080. ...,González-Parra G,infect dis model,2024,2024/07/11,PMC11233876,NaN,10.1016/j.idm.2024.05.005
...,...,...,...,...,...,...,...,...,...,...,...
75,39180016,Impacts of blended learning with BOPPPS model ...,"Li S, Wei W, Li X, Ma L, Li Q, Sun X, Chen X.",BMC Med Educ. 2024 Aug 23;24(1):914. doi: 10.1...,Li S,bmc med educ,2024,2024/08/23,PMC11344447,NaN,10.1186/s12909-024-05917-x
76,39045688,Symptom propagation in respiratory pathogens o...,"Asplin P, Mancy R, Finnie T, Cumming F, Keelin...",J R Soc Interface. 2024 Jul;21(216):20240009. ...,Asplin P,j r soc interface,2024,2024/07/24,PMC11267474,NaN,10.1098/rsif.2024.0009
77,39596075,Zebrafish (Danio rerio) as a Model System to I...,"Franza M, Varricchio R, Alloisio G, De Simone ...",Int J Mol Sci. 2024 Nov 8;25(22):12008. doi: 1...,Franza M,int j mol sci,2024,2024/11/27,PMC11593600,NaN,10.3390/ijms252212008
78,38819720,The Value of Flexible Vaccine Manufacturing Ca...,"McElwee F, Newall A.",Pharmacoeconomics. 2024 Jul;42(Suppl 2):187-19...,McElwee F,pharmacoeconomics,2024,2024/05/31,PMC11230966,NaN,10.1007/s40273-024-01396-6


In [92]:
# Extract and map journal abbreviations to full journal names and subject areas.
# This script processes a dataset of PubMed publications where journals are represented
# in their abbreviated forms. It retrieves the corresponding full journal names and
# associated subject areas, enhancing the dataset's readability and analytical potential.

def get_journal_info(journal_name):
    """Fetch ISSN and full journal title from PubMed using Entrez."""
    try:
        search_term = f"{journal_name}[SO]"
        journal_query = Entrez.esearch(db="pubmed", term=search_term)
        record = Entrez.read(journal_query)
        
        if not record["IdList"]:
            return None, None
        
        journal_id = record["IdList"][0]
        summary = Entrez.esummary(db="pubmed", id=journal_id)
        summary_data = Entrez.read(summary)
        
        issn = summary_data[0].get("ISSN", None)
        full_title = summary_data[0].get("FullJournalName", journal_name)  # Default to input name
        
        return issn, full_title
    except:
        return None, journal_name

# Apply function to get journal details
papers["ISSN"], papers["Journal/Book"] = zip(*papers["Journal/Book"].apply(get_journal_info))
papers

/opt/anaconda3/lib/python3.12/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI,ISSN
0,39142261,Data-driven mathematical modeling approaches f...,"Demongeot J, Magal P.",Phys Life Rev. 2024 Sep;50:166-208. doi: 10.10...,Demongeot J,Physics of life reviews,2024,2024/08/14,NaN,NaN,10.1016/j.plrev.2024.08.004,1571-0645
1,38421182,Pathogenesis of viral infections during pregnancy,"Creisher PS, Klein SL.",Clin Microbiol Rev. 2024 Jun 13;37(2):e0007323...,Creisher PS,Clinical microbiology reviews,2024,2024/02/29,PMC11237665,NaN,10.1128/cmr.00073-23,0893-8512
2,38230499,Recent advances in breast cancer cell line res...,"Sharma MP, Shukla S, Misra G.",Int J Cancer. 2024 May 15;154(10):1683-1693. d...,Sharma MP,International journal of cancer,2024,2024/01/17,NaN,NaN,10.1002/ijc.34849,0020-7136
3,39270616,Early mathematical models of COVID-19 vaccinat...,"Burch E, Khan SA, Stone J, Asgharzadeh A, Dawe...",Public Health. 2024 Nov;236:207-215. doi: 10.1...,Burch E,BMC public health,2024,2024/09/13,NaN,NaN,10.1016/j.puhe.2024.07.029,
4,38988830,Learning from the COVID-19 pandemic: A systema...,"González-Parra G, Mahmud MS, Kadelka C.",Infect Dis Model. 2024 May 15;9(4):1057-1080. ...,González-Parra G,Infectious Disease Modelling,2024,2024/07/11,PMC11233876,NaN,10.1016/j.idm.2024.05.005,2468-2152
...,...,...,...,...,...,...,...,...,...,...,...,...
75,39180016,Impacts of blended learning with BOPPPS model ...,"Li S, Wei W, Li X, Ma L, Li Q, Sun X, Chen X.",BMC Med Educ. 2024 Aug 23;24(1):914. doi: 10.1...,Li S,BMC medical education,2024,2024/08/23,PMC11344447,NaN,10.1186/s12909-024-05917-x,
76,39045688,Symptom propagation in respiratory pathogens o...,"Asplin P, Mancy R, Finnie T, Cumming F, Keelin...",J R Soc Interface. 2024 Jul;21(216):20240009. ...,Asplin P,"Journal of the Royal Society, Interface",2024,2024/07/24,PMC11267474,NaN,10.1098/rsif.2024.0009,1742-5689
77,39596075,Zebrafish (Danio rerio) as a Model System to I...,"Franza M, Varricchio R, Alloisio G, De Simone ...",Int J Mol Sci. 2024 Nov 8;25(22):12008. doi: 1...,Franza M,International journal of molecular sciences,2024,2024/11/27,PMC11593600,NaN,10.3390/ijms252212008,
78,38819720,The Value of Flexible Vaccine Manufacturing Ca...,"McElwee F, Newall A.",Pharmacoeconomics. 2024 Jul;42(Suppl 2):187-19...,McElwee F,PharmacoEconomics,2024,2024/05/31,PMC11230966,NaN,10.1007/s40273-024-01396-6,1170-7690


In [109]:
# Normalize journal names between the PubMed and SJR datasets using fuzzy matching.
# This step is crucial for accurately linking impact factors from the SJR dataset to PubMed publications,
# as journal names may differ slightly due to abbreviations or variations in spelling.

from rapidfuzz import process

def find_closest_match(journal_name, journal_list):
    """Find the closest journal match using fuzzy matching."""
    result = process.extractOne(journal_name, journal_list)
    if result:
        match, score = result[0], result[1]  # Unpacking only first two values
        return match if score > 80 else None
    return None

# Apply fuzzy matching
papers["Matched_Journal"] = papers["Journal/Book"].apply(lambda x: find_closest_match(x, sjr_data["Title"].tolist()))

# Merge with SJR data
papers = papers.merge(sjr_data, left_on="Matched_Journal", right_on="Title", how="left")

# Rename columns
papers.rename(columns={"SJR": "Impact_Factor", "Areas": "Subject_Area"}, inplace=True)

# Handle missing values
papers["Impact_Factor"].fillna("Not Available", inplace=True)
papers["Subject_Area"].fillna("Unknown", inplace=True)
papers


/var/folders/v3/kx2f1_7n79b9wzpp4g0856z40000gn/T/ipykernel_13070/3689257679.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  papers["Impact_Factor"].fillna("Not Available", inplace=True)
/var/folders/v3/kx2f1_7n79b9wzpp4g0856z40000gn/T/ipykernel_13070/3689257679.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

,PMID,Title_x,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,...,Ref. / Doc.,%Female,Overton,SDG,Country,Region,Publisher,Coverage,Categories,Subject_Area
0,39142261,Data-driven mathematical modeling approaches f...,"Demongeot J, Magal P.",Phys Life Rev. 2024 Sep;50:166-208. doi: 10.10...,Demongeot J,Physics of life reviews,2024,2024/08/14,NaN,NaN,...,"36,54","27,87",0.0,16.0,Netherlands,Western Europe,Elsevier B.V.,2004-2023,Agricultural and Biological Sciences (miscella...,Agricultural and Biological Sciences; Computer...
1,38421182,Pathogenesis of viral infections during pregnancy,"Creisher PS, Klein SL.",Clin Microbiol Rev. 2024 Jun 13;37(2):e0007323...,Creisher PS,Clinical microbiology reviews,2024,2024/02/29,PMC11237665,NaN,...,"228,46","30,00",0.0,17.0,United States,Northern America,American Society for Microbiology,1988-2023,Epidemiology (Q1); Immunology and Microbiology...,Immunology and Microbiology; Medicine
2,38230499,Recent advances in breast cancer cell line res...,"Sharma MP, Shukla S, Misra G.",Int J Cancer. 2024 May 15;154(10):1683-1693. d...,Sharma MP,International journal of cancer,2024,2024/01/17,NaN,NaN,...,"45,27","45,98",10.0,394.0,United States,Northern America,Wiley-Liss Inc.,1966-2023,Cancer Research (Q1); Oncology (Q1),"Biochemistry, Genetics and Molecular Biology; ..."
3,39270616,Early mathematical models of COVID-19 vaccinat...,"Burch E, Khan SA, Stone J, Asgharzadeh A, Dawe...",Public Health. 2024 Nov;236:207-215. doi: 10.1...,Burch E,BMC public health,2024,2024/09/13,NaN,NaN,...,"36,59","50,03",12.0,261.0,Netherlands,Western Europe,Elsevier B.V.,"1888-1913, 1915-2023","Medicine (miscellaneous) (Q1); Public Health, ...",Medicine
4,38988830,Learning from the COVID-19 pandemic: A systema...,"González-Parra G, Mahmud MS, Kadelka C.",Infect Dis Model. 2024 May 15;9(4):1057-1080. ...,González-Parra G,Infectious Disease Modelling,2024,2024/07/11,PMC11233876,NaN,...,"40,71","33,42",0.0,73.0,China,Asiatic Region,KeAi Communications Co.,2016-2023,Applied Mathematics (Q1); Health Policy (Q1); ...,Mathematics; Medicine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,39180016,Impacts of blended learning with BOPPPS model ...,"Li S, Wei W, Li X, Ma L, Li Q, Sun X, Chen X.",BMC Med Educ. 2024 Aug 23;24(1):914. doi: 10.1...,Li S,BMC medical education,2024,2024/08/23,PMC11344447,NaN,...,"25,06","58,86",0.0,51.0,United Kingdom,Western Europe,Wiley-Blackwell Publishing Ltd,1966-2023,Education (Q1); Medicine (miscellaneous) (Q1),Medicine; Social Sciences
78,39045688,Symptom propagation in respiratory pathogens o...,"Asplin P, Mancy R, Finnie T, Cumming F, Keelin...",J R Soc Interface. 2024 Jul;21(216):20240009. ...,Asplin P,"Journal of the Royal Society, Interface",2024,2024/07/24,PMC11267474,NaN,...,"61,05","29,20",0.0,44.0,United Kingdom,Western Europe,The Royal Society,2004-2023,Biochemistry (Q1); Bioengineering (Q1); Biomed...,"Biochemistry, Genetics and Molecular Biology; ..."
79,39596075,Zebrafish (Danio rerio) as a Model System to I...,"Franza M, Varricchio R, Alloisio G, De Simone ...",Int J Mol Sci. 2024 Nov 8;25(22):12008. doi: 1...,Franza M,International journal of molecular sciences,2024,2024/11/27,PMC11593600,NaN,...,"78,63","48,27",21.0,7012.0,Switzerland,Western Europe,Multidisciplinary Digital Publishing Institute...,2000-2023,Computer Science Applications (Q1); Inorganic ...,"Biochemistry, Genetics and Molecular Biology; ..."
80,38819720,The Value of Flexible Vaccine Manufacturing Ca...,"McElwee F, Newall A.",Pharmacoeconomics. 2024 Jul;42(Suppl 2):187-19...,McElwee F,PharmacoEconomics,2024,2024/05/31,PMC11230966,NaN,...,"54,57","46,30",8.0,48.0,United Kingdom,Western Europe,Adis International Ltd,1992-2023,Health Policy (Q1); Pharmacology (Q1); Public ...,"Medicine; Pharmacology, Toxicology and Pharmac..."


In [131]:
# Retrieve citation counts and publication dates from the Scopus API using an API key.
# Consider allowing users to provide their own API keys for flexibility and security,
# rather than embedding a personal key in the script.
# This data is used for calculating citations per year in subsequent analysis.

SCOPUS_API_KEY = "aa2904d42e8be95313e0134ba81a7890"

def fetch_metadata(title):
    """Fetch citation count, publication year, and DOI from Scopus."""
    query = f"TITLE(\"{title}\")"
    encoded_query = requests.utils.quote(query)
    url = f"https://api.elsevier.com/content/search/scopus?query={encoded_query}"
    
    headers = {"X-ELS-APIKey": SCOPUS_API_KEY}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        return None, None, None
    
    data = response.json()
    entry = data.get("search-results", {}).get("entry", [{}])[0]

    citation_count = entry.get("citedby-count", None)
    pub_date = entry.get("prism:coverDate", None)
    doi = entry.get("prism:doi", None)
    
    return citation_count, pub_date, doi

# Apply function to each paper
papers[["Citations", "Publication_Date", "DOI"]] = papers["Title_x"].apply(lambda x: fetch_metadata(x)).apply(pd.Series)


In [135]:
# computing citations per year

from datetime import datetime
import pandas as pd

# Ensure the "Citations" column is numeric
papers["Citations"] = pd.to_numeric(papers["Citations"], errors="coerce")

# Extract Year and Month from "Publication_Date"
papers["Publication_Date"] = pd.to_datetime(papers["Publication_Date"], errors="coerce")
papers["Publication_Year"] = papers["Publication_Date"].dt.year
papers["Publication_Month"] = papers["Publication_Date"].dt.month

# Get current year and month
current_year = datetime.now().year
current_month = datetime.now().month

# Function to calculate citation per year
def calculate_citation_per_year(row):
    """Calculate citation per year considering full years and partial years based on publication month."""
    if pd.isna(row["Citations"]) or pd.isna(row["Publication_Year"]):
        return None  # If citations or year are missing, return None
    
    full_years_since_publication = current_year - row["Publication_Year"] - 1  # Full years since publication
    
    # Fraction of the publication year
    if pd.notna(row["Publication_Month"]):
        publication_fraction = (12 - row["Publication_Month"] + 1) / 12
    else:
        publication_fraction = 1  # Assume full year if month is missing
    
    # Fraction of the current year
    current_year_fraction = current_month / 12
    
    # Total effective years since publication
    total_years_since_publication = full_years_since_publication + publication_fraction + current_year_fraction
    
    if total_years_since_publication > 0:
        return row["Citations"] / total_years_since_publication
    else:
        return row["Citations"]  # If published this year, use raw citation count

# Apply the function to compute citation per year
papers["Citation_Per_Year"] = papers.apply(calculate_citation_per_year, axis=1)


In [146]:
# Normalize citation counts per year and impact factors within each subject field using min-max scaling.
# This ensures metrics are on a comparable scale, facilitating fair comparisons across different fields.

# Data Source Considerations:
#   - Currently using the SJR Scimago dataset.
#     - Issue: Grouped subject fields may combine similar areas, reducing granularity.
#   - Alternative: Scopus Sources (https://www.scopus.com/sources.uri).
#     - Potential: Allows for normalization of impact factors at the individual journal level.
#     - Limitation: Scopus currently limits results to 1000 journals, while the total journal list is ~45,000.
#     - Further research is needed to determine if the Scopus API can be used to retrieve all journal data.


from sklearn.preprocessing import MinMaxScaler

papers["Impact_Factor"] = papers["Impact_Factor"].replace("Not Available", np.nan)

# Ensure Impact_Factor is numeric (replace "," with "." and convert to float)
papers["Impact_Factor"] = papers["Impact_Factor"].astype(str).str.replace(",", ".").astype(float, errors="ignore")

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

def normalize_column(group, column):
    """Normalize a column within each Subject_Area."""
    if group[column].notna().sum() > 1:  # Only normalize if more than one non-null value exists
        group[f"Field_Normalized_{column}"] = scaler.fit_transform(group[[column]])
    else:
        group[f"Field_Normalized_{column}"] = 1  # If all values are identical, set to 1
    return group

# Apply normalization for each field within "Subject_Area"
for col in ["Citations", "Citation_Per_Year", "Impact_Factor"]:
    papers = papers.groupby("Subject_Area", group_keys=False).apply(lambda g: normalize_column(g, col))



/var/folders/v3/kx2f1_7n79b9wzpp4g0856z40000gn/T/ipykernel_13070/3869724606.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  papers = papers.groupby("Subject_Area", group_keys=False).apply(lambda g: normalize_column(g, col))
/var/folders/v3/kx2f1_7n79b9wzpp4g0856z40000gn/T/ipykernel_13070/3869724606.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  papers = papers.groupby("Subject_Area", group_keys=F

In [150]:
# Calculate an inclusion score for each paper based on normalized impact factor and normalized citations per year.
# The inclusion score is a weighted average, ranging from 0 to 1, reflecting the combined importance of these metrics.

# Weighting:
#   - Weights can be fixed at 0.5 for both impact factor and citations per year, providing equal importance.
#   - Alternatively, users can specify custom weights to reflect their specific priorities.

# Priority Categorization:
#   - Based on the calculated inclusion score, papers are categorized into priority levels:
#     - High: Inclusion score > 0.75
#     - Medium: 0.5 <= Inclusion score <= 0.75
#     - Low: Inclusion score < 0.5
#   - A 'Priority' column is added to the dataset to reflect these categories, facilitating easy identification of high-priority papers.


# Define weights
alpha, beta = 0.5, 0.5  # Adjustable

# Compute Inclusion Score
papers["Inclusion_Score"] = (alpha * papers["Field_Normalized_Citation_Per_Year"]) + (beta * papers["Field_Normalized_Impact_Factor"])

# Categorize into Priority Levels
papers["Inclusion_Category"] = pd.cut(
    papers["Inclusion_Score"],
    bins=[-np.inf, 0.5, 0.75, np.inf],
    labels=["Low Priority", "Medium Priority", "High Priority"]
)


In [158]:
# Alternative priority labeling using a percentile-based cutoff.
# Papers are categorized into "High Priority" and "Low Priority" based on their inclusion score relative to the 70th percentile.

# Labeling Criteria:
#   - High Priority: Papers with inclusion scores exceeding the 70th percentile of all inclusion scores.
#   - Low Priority: Papers with inclusion scores at or below the 70th percentile.

# This approach provides a dynamic cutoff, adapting to the distribution of inclusion scores in the dataset.
# It can be more robust than fixed thresholds (e.g., 0.75) when dealing with datasets with varying score distributions.


# Compute 70th percentile cutoff
cutoff = np.nanpercentile(papers["Inclusion_Score"], 70)

# Assign final priority category
papers["Inclusion_Category_by_percentile"] = np.where(papers["Inclusion_Score"] >= cutoff, "High Priority", "Low Priority")
